# small language model test implementation

In [157]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import einops
from torch.utils.data import Dataset
import pandas as pd
import os
CACHE_DIR = "/data/zejiaqi/huggingface-cache"
os.environ["HF_HOME"] = CACHE_DIR
from datasets import load_dataset

In [158]:
ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", cache_dir=CACHE_DIR)

In [159]:
SEP = "\n<|endoftext|>\n"   # documents are independent; don't let merges run across them

shuffled = ds.shuffle(seed=42)
train_docs = shuffled.select(range(10000))["text"]
held_docs = shuffled.select(range(10000, 10200))["text"]   # disjoint: never seen in training

train_text = SEP.join(train_docs)
held_text = SEP.join(held_docs)
print(f"train {len(train_text.encode())/1e6:.1f} MB | held-out {len(held_text.encode())/1e6:.1f} MB")

train 47.1 MB | held-out 1.2 MB


In [160]:
import re, json
from collections import Counter

SPLIT_PATTERN = re.compile(r"\s*\S+|\s+")


class SimpleTokenizer:
    def __init__(self, vocab_size: int = 10000):
        self.vocab_size = vocab_size
        self.vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.merge_rules: dict[tuple[int, int], int] = {}
        self._cache: dict[str, list[int]] = {}

    def train_tokenizer(self, text: str) -> None:
        freqs = Counter(SPLIT_PATTERN.findall(text))
        chunk_ids = [list(chunk.encode("utf-8")) for chunk in freqs]
        weights = list(freqs.values())

        counts: dict[tuple[int, int], int] = {}
        where: dict[tuple[int, int], set[int]] = {}
        for idx, ids in enumerate(chunk_ids):
            self._index_chunk(idx, ids, weights[idx], counts, where, +1)

        for _ in range(self.vocab_size - 256):
            if not counts:
                break
            pair = max(counts, key=counts.get)
            new_id = len(self.vocab)
            self.merge_rules[pair] = new_id
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]

            for idx in list(where.get(pair, ())):
                ids = chunk_ids[idx]
                self._index_chunk(idx, ids, weights[idx], counts, where, -1)
                ids = self._merge(ids, pair, new_id)
                chunk_ids[idx] = ids
                self._index_chunk(idx, ids, weights[idx], counts, where, +1)
        self._cache.clear()

    @staticmethod
    def _index_chunk(idx, ids, weight, counts, where, sign) -> None:
        """Add (sign=+1) or remove (sign=-1) one chunk's pair statistics."""
        for pair in zip(ids, ids[1:]):
            total = counts.get(pair, 0) + sign * weight
            if total <= 0:
                counts.pop(pair, None)
                where.pop(pair, None)
            else:
                counts[pair] = total
                if sign > 0:
                    where.setdefault(pair, set()).add(idx)

    def _merge(self, text: list[int], pair: tuple[int, int], new_id: int) -> list[int]:
        """Replace every non-overlapping occurrence of `pair` with `new_id`."""
        new_text = []
        i = 0
        while i < len(text):
            if text[i] == pair[0] and i < len(text) - 1 and text[i + 1] == pair[1]:
                new_text.append(new_id)
                i += 2
            else:
                new_text.append(text[i])
                i += 1
        return new_text

    def _encode_chunk(self, chunk: str) -> list[int]:
        cached = self._cache.get(chunk)
        if cached is not None:
            return cached
        ids = list(chunk.encode("utf-8"))
        while len(ids) >= 2:
            best = min(zip(ids, ids[1:]), key=lambda p: self.merge_rules.get(p, float("inf")))
            if best not in self.merge_rules:
                break
            ids = self._merge(ids, best, self.merge_rules[best])
        self._cache[chunk] = ids
        return ids

    def encode(self, text: str) -> list[int]:
        out: list[int] = []
        for chunk in SPLIT_PATTERN.findall(text):
            out.extend(self._encode_chunk(chunk))
        return out

    def decode(self, tokens: list[int]) -> str:
        return b"".join(self.vocab[t] for t in tokens).decode("utf-8", errors="replace")

    def save(self, path: str) -> None:
        with open(path, "w") as f:
            json.dump({"vocab_size": self.vocab_size,
                       "merges": [[a, b, i] for (a, b), i in self.merge_rules.items()]}, f)

    @classmethod
    def load(cls, path: str) -> "SimpleTokenizer":
        with open(path) as f:
            data = json.load(f)
        tok = cls(vocab_size=data["vocab_size"])
        for a, b, new_id in sorted(data["merges"], key=lambda m: m[2]):
            tok.merge_rules[(a, b)] = new_id
            tok.vocab[new_id] = tok.vocab[a] + tok.vocab[b]
        return tok

In [161]:
tokenizer = SimpleTokenizer(vocab_size=4096)

In [162]:
# %time tokenizer.train_tokenizer(train_text)
# tokenizer.save("tokenizer.json")
tokenizer = SimpleTokenizer.load("tokenizer.json")

In [163]:
s = "hello world — café 🙂"
assert tokenizer.decode(tokenizer.encode(s)) == s
assert tokenizer.decode(tokenizer.encode(held_text)) == held_text
assert tokenizer.encode("") == [] and tokenizer.encode("a") == [97]

print("vocab entries:", len(tokenizer.vocab))
print("compression:", round(len(held_text.encode()) / len(tokenizer.encode(held_text)), 2), "bytes/token")

vocab entries: 4096
compression: 3.43 bytes/token


In [ ]:
import numpy as np

device = torch.device('cuda:3') if torch.cuda.is_available() else torch.device('cpu')
BLOCK_SIZE = 512

# The 200k-doc corpus is encoded + cached by train_slm.py (parallel workers, ~1 min).
# Load the memmap here rather than re-encoding in-notebook.
DATA = "/data/zejiaqi/model_tests/slm/data"
corpus = torch.from_numpy(np.load(f"{DATA}/train_full_200000.npy").astype(np.int64)).to(device)
corpus_test = torch.from_numpy(np.load(f"{DATA}/val_full_5000.npy").astype(np.int64)).to(device)
print(f"train {len(corpus)/1e6:.1f}M tokens | val {len(corpus_test)/1e6:.1f}M tokens")

In [165]:
def get_batch(corpus: torch.Tensor, batch_size: int = 32, block_size: int = 1024) -> tuple[torch.Tensor, torch.Tensor]:
    """
    returns x, y of (B, T)
    """
    idx = torch.randint(0, len(corpus) - block_size, (batch_size,))
    x = torch.stack([corpus[i:i + block_size] for i in idx])
    y = torch.stack([corpus[i + 1:i + 1 + block_size] for i in idx])
    return x, y

In [166]:

class BigramPredictor(nn.Module):
    def __init__(self, vocab_size: int = 4096):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, x, targets=None):
        logits = self.embedding(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                einops.rearrange(logits, "b t v -> (b t) v"),
                einops.rearrange(targets, "b t -> (b t)"),
            )
            
        return logits, loss

In [ ]:
class MultiheadSelfAttention(nn.Module):
    def __init__(self, hidden_dim: int = 768, num_heads: int = 12, block_size: int = 1024):
        super().__init__()
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.scale = self.head_dim ** 0.5

    def forward(self, x):
        q_full = self.q_proj(x)
        k_full = self.k_proj(x)
        v_full = self.v_proj(x)

        q_split = einops.rearrange(q_full, "b t (h d) -> b h t d", h=self.num_heads)
        k_split = einops.rearrange(k_full, "b t (h d) -> b h t d", h=self.num_heads)
        v_split = einops.rearrange(v_full, "b t (h d) -> b h t d", h=self.num_heads)

        scores_full = einops.einsum(q_split, k_split, "b h i d, b h j d -> b h i j")
        scores_full = scores_full / self.scale

        T = scores_full.size(-1)
        scores = torch.masked_fill(scores_full, self.tril[:T, :T] == 0, float('-inf'))  # ty:ignore[not-subscriptable]
        scores = F.softmax(scores, dim=-1)

        v_to_add = einops.einsum(scores, v_split, "b h q k, b h k d -> b h q d")
        v_concat = einops.rearrange(v_to_add, "b h t d -> b t (h d)")

        return self.out_proj(v_concat)


class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim: int = 768, num_heads: int = 12, block_size: int = 1024):
        super().__init__()
        self.attn = MultiheadSelfAttention(hidden_dim, num_heads, block_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim),
        )
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        x = self.attn(self.ln1(x)) + x
        x = self.ffn(self.ln2(x)) + x
        return x


class JLM(nn.Module):
    def __init__(self, vocab_size: int = 4096, hidden_dim: int = 768,
                 num_heads: int = 12, n_layer: int = 12, block_size: int = 1024):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.pos_embed = nn.Embedding(block_size, hidden_dim)
        self.blocks = nn.Sequential(*[
            TransformerBlock(hidden_dim, num_heads, block_size) for _ in range(n_layer)
        ])
        self.norm = nn.LayerNorm(hidden_dim)
        self.lm_head = nn.Linear(hidden_dim, vocab_size, bias=False)

    def forward(self, x, targets=None):
        B, T = x.shape
        tokens = self.embedding(x)
        positions = self.pos_embed(torch.arange(T, device=x.device))
        stream = tokens + positions

        transformed_stream = self.norm(self.blocks(stream))
        logits = self.lm_head(transformed_stream)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                einops.rearrange(logits, "b t v -> (b t) v"),
                einops.rearrange(targets, "b t -> (b t)"),
            )

        return logits, loss

In [ ]:
model = JLM(block_size=512, hidden_dim=768, num_heads=12, n_layer=12).to(device)
print(f"model params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M")

In [ ]:
import math
from tqdm.auto import tqdm

# NOTE: train_slm.py is the active background runner for the real 100M run (on cuda:3,
# logging to train.log, checkpoints in checkpoints/). Only run this cell if that is NOT
# running, or you'll contend for the same GPU.

MAX_STEPS, WARMUP, BATCH = 30_000, 500, 32
LR, MIN_LR = 3e-4, 3e-5

def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    prog = (step - WARMUP) / (MAX_STEPS - WARMUP)
    return MIN_LR + 0.5 * (LR - MIN_LR) * (1 + math.cos(math.pi * prog))

@torch.no_grad()
def eval_loss(iters=50):
    model.eval()
    losses = [model(*get_batch(corpus_test, BATCH, BLOCK_SIZE))[1].item() for _ in range(iters)]
    model.train()
    return sum(losses) / len(losses)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1, betas=(0.9, 0.95))

pbar = tqdm(range(MAX_STEPS))
for step in pbar:
    for g in optimizer.param_groups:
        g["lr"] = lr_at(step)
    x, y = get_batch(corpus, BATCH, BLOCK_SIZE)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        logits, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step % 500 == 0:
        pbar.set_postfix(train=f"{loss.item():.3f}", val=f"{eval_loss():.3f}")

In [180]:
@torch.no_grad()
def generate(model, prompt="\n<|endoftext|>\n", max_new_tokens=200, temperature=1.0):
    model.eval()
    ids = torch.tensor([tokenizer.encode(prompt)], device=device)   # (1, T)
    for _ in range(max_new_tokens):
        logits, _ = model(ids)                    # (1, T, vocab_size)
        logits = logits[:, -1, :] / temperature   # (1, vocab_size): last step only
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)   # sample, not argmax
        ids = torch.cat([ids, next_id], dim=1)
    model.train()
    return tokenizer.decode(ids[0].tolist())

print(generate(model))



<|endoftext|>
disapproximately 800 million squaff exchange for smallholders. It is literally functioning only with rampels,” says January, in Juan, World.
But, less it was to begin now and member days of one state each year a year. Planet there are special punds that are not just 100 censities that are traded domesticated to cues from the U.S. — by proprietors’ energy and decreasing the dozen agencies and tides, grew longer than in further refugees.
Because the five treatments are significant from cultivation, all sector changes have occurred on without (in the twenty-live by 5 million ties people) are analysed.
“Our study has shown that coined productivity to cider with the components of energy are activated,” says Maria on the Department of Agriculture at Orar Recycling
